# Course 8: Privacy-Preserving Machine Learning
## Session 5: Federated learning

**Goal:** simulate ten hospitals with skewed data, write FedAvg and FedProx by hand, and measure what the skew costs.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Ten hospitals
A Dirichlet split controls the skew. A small `alpha` means each hospital sees only a few of the classes.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, copy

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
_d = load_digits()
Xtr, Xte, ytr, yte = train_test_split(torch.tensor(_d.data / 16.0, dtype=torch.float32), torch.tensor(_d.target),
                                      test_size=0.25, random_state=0, stratify=_d.target)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

def dirichlet_split(y, n_clients, alpha, rng):
    parts = [[] for _ in range(n_clients)]
    for c in range(10):
        idx = rng.permutation(np.flatnonzero(y == c))
        cuts = (np.cumsum(rng.dirichlet(alpha * np.ones(n_clients)))[:-1] * len(idx)).astype(int)
        for p, chunk in zip(parts, np.split(idx, cuts)): p.extend(chunk.tolist())
    return [np.array(p) for p in parts]

rng = np.random.default_rng(0)
iid = np.array_split(rng.permutation(len(ytr)), 10)
skewed = dirichlet_split(ytr.numpy(), 10, alpha=0.1, rng=rng)
for h in range(3): print(f"hospital {h}: {len(skewed[h]):3d} patients, class counts {np.bincount(ytr[skewed[h]].numpy(), minlength=10)}")

### 2. FedAvg and FedProx
Each round, every hospital starts from the shared model, trains locally, and sends its weights back. The server averages them, weighted by how much data each hospital has. FedProx adds a penalty `mu/2 * |w - w_global|²` to the local loss, which stops a hospital drifting too far.

In [ ]:
def new_model():
    return nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))

def local_train(global_model, idx, epochs, mu, lr=0.1):
    model = copy.deepcopy(global_model); opt = torch.optim.SGD(model.parameters(), lr=lr)
    g_params = [p.detach().clone() for p in global_model.parameters()]
    for _ in range(epochs):
        for batch in torch.tensor(np.random.default_rng(0).permutation(idx)).split(32):
            loss = F.cross_entropy(model(Xtr[batch]), ytr[batch])
            if mu > 0: loss = loss + mu / 2 * sum(((p - g) ** 2).sum() for p, g in zip(model.parameters(), g_params))
            opt.zero_grad(); loss.backward(); opt.step()
    return model.state_dict()

def federated(parts, rounds=20, local_epochs=5, mu=0.0, seed=0):
    torch.manual_seed(seed); glob = new_model(); history = []
    sizes = np.array([len(p) for p in parts], dtype=float); w = sizes / sizes.sum()
    for r in range(rounds):
        states = [local_train(glob, p, local_epochs, mu) for p in parts if len(p) > 0]
        ws = [wi for wi, p in zip(w, parts) if len(p) > 0]
        glob.load_state_dict({k: sum(wi * s[k] for wi, s in zip(ws, states)) / sum(ws) for k in states[0]})
        history.append((glob(Xte).argmax(1) == yte).float().mean().item())
    return history

h_iid = federated(iid)
h_avg = federated(skewed)
h_prox = federated(skewed, mu=0.1)
for name, h in [("FedAvg, IID data", h_iid), ("FedAvg, skewed data", h_avg), ("FedProx, skewed data", h_prox)]:
    print(f"{name:22s} round 1: {h[0]:.3f}   round 5: {h[4]:.3f}   round 20: {h[-1]:.3f}")
assert h_iid[4] > h_avg[4]

Read the round-5 column: skew slows learning down. Whether FedProx helps depends on `mu`, the number of local epochs and the seed. Report what you measure, not what the paper title promises.

### 3. What each hospital could do alone

In [ ]:
def alone(idx):
    torch.manual_seed(0); m = new_model(); opt = torch.optim.SGD(m.parameters(), lr=0.1)
    for _ in range(100):
        for batch in torch.tensor(idx).split(32):
            loss = F.cross_entropy(m(Xtr[batch]), ytr[batch]); opt.zero_grad(); loss.backward(); opt.step()
    return (m(Xte).argmax(1) == yte).float().mean().item()
solo = [alone(p) for p in skewed if len(p) > 0]
print("skewed hospitals training alone: mean accuracy %.3f (best %.3f) | federated: %.3f" % (np.mean(solo), max(solo), h_avg[-1]))
assert h_avg[-1] > np.mean(solo)

### 4. Is this private?
Not yet. The data stays in each hospital, but the weight updates still carry information about it (session 6 rebuilds an image from them). Federated learning is a way to organize training. Privacy needs secure aggregation, differential privacy, or both on top.

### 5. The same simulation in Flower (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install "flwr[simulation]"`. Flower's API has changed between versions; follow the current quickstart and adapt.
import flwr as fl

class Hospital(fl.client.NumPyClient):
    def __init__(self, idx): self.idx, self.model = idx, new_model()
    def set(self, params):
        self.model.load_state_dict({k: torch.tensor(v) for k, v in zip(self.model.state_dict(), params)})
    def get(self): return [v.numpy() for v in self.model.state_dict().values()]
    def fit(self, params, config):
        self.set(params)
        self.model.load_state_dict(local_train(self.model, self.idx, epochs=5, mu=config.get("proximal_mu", 0.0)))
        return self.get(), len(self.idx), {}
    def evaluate(self, params, config):
        self.set(params); acc = (self.model(Xte).argmax(1) == yte).float().mean().item()
        return 0.0, len(Xte), {"accuracy": acc}

# strategy = fl.server.strategy.FedAvg()   or   fl.server.strategy.FedProx(proximal_mu=0.1)
# Start a simulation with 10 clients built from `skewed`, run 20 rounds, and compare with the numbers above.

### TODO
1. Sweep alpha in [0.05, 0.1, 0.5, 5] and plot final accuracy.
2. Sweep local epochs in [1, 5, 20] on the skewed split. When does more local work start to hurt?
3. Let only 3 random hospitals take part in each round. What changes?